# Apache Iceberg — Controle de produtos de uma loja

Este notebook demonstra o uso do Apache Iceberg com PySpark para
armazenar e modificar dados de uma loja.

O cenário possui duas tabelas: categorias e produtos. Uma categoria
pode ter vários produtos, e cada produto pertence a uma categoria.

Os dados são fictícios e foram criados pelo grupo. Utilizamos os mesmos
registros do notebook Delta para demonstrar INSERT, UPDATE e DELETE.

## 1. Preparação do ambiente

Nesta etapa, iniciamos uma sessão Spark com suporte ao Apache Iceberg.

Configuramos um catálogo chamado local, do tipo hadoop, para organizar
as tabelas e armazenar seus arquivos na pasta data/iceberg.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession

pasta_iceberg = (Path("..") / "data" / "iceberg").resolve()
pasta_iceberg.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("Loja - Apache Iceberg")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.local.type", "hadoop")
    .config(
        "spark.sql.catalog.local.warehouse",
        pasta_iceberg.as_posix()
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Versão do Spark:", spark.version)
spark.sql("SHOW NAMESPACES IN local").show()

26/10/05 21:16:05 WARN Utils: Your hostname, YasminNitroV15 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/05 21:16:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/bfontanayasmin/projetos/trabalhoBancodeDados/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/bfontanayasmin/.ivy2/cache
The jars for the packages stored in: /home/bfontanayasmin/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d378377f-8cf4-4168-b5da-6e720eaf2ba4;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
:: resolution report :: resolve 88ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :

Versão do Spark: 3.5.3
+------------+
|   namespace|
+------------+
|loja_iceberg|
+------------+



### Limpeza do exemplo

Esta etapa remove as tabelas e os arquivos de uma execução anterior,
permitindo iniciar novamente a demonstração.

DROP TABLE IF EXISTS remove as tabelas quando elas existem.
A opção PURGE também solicita a exclusão dos arquivos das tabelas.

Depois, removemos e recriamos a pasta data/iceberg para começar
com o ambiente vazio.

In [2]:
import shutil

raiz_projeto = Path("..").resolve()

if not (raiz_projeto / "pyproject.toml").is_file():
    raise RuntimeError("Execute este notebook dentro da pasta notebooks.")

spark.catalog.clearCache()

spark.sql("DROP TABLE IF EXISTS local.loja_iceberg.produtos PURGE")
spark.sql("DROP TABLE IF EXISTS local.loja_iceberg.categorias PURGE")

pasta_iceberg = raiz_projeto / "data" / "iceberg"

if pasta_iceberg.exists():
    shutil.rmtree(pasta_iceberg)

pasta_iceberg.mkdir(parents=True, exist_ok=True)

print("Limpeza concluída. O exemplo Iceberg pode ser iniciado novamente.")

Limpeza concluída. O exemplo Iceberg pode ser iniciado novamente.


## 2. Estrutura das tabelas

Utilizamos o mesmo modelo do notebook Delta.

| Tabela | Campo | Tipo |
|---|---|---|
| categorias | id_categoria | INT |
| categorias | nome_categoria | STRING |
| produtos | id_produto | INT |
| produtos | nome_produto | STRING |
| produtos | preco | DECIMAL(10,2) |
| produtos | estoque | INT |
| produtos | id_categoria | INT |

### Modelo entidade-relacionamento

![Modelo ER da loja](../docs/assets/modelo_er_loja.png)

Uma categoria pode possuir vários produtos, e cada produto pertence
a uma categoria. O campo id_categoria permite relacionar as tabelas.

Os identificadores representam os papéis de chave no modelo.
Neste exemplo, não definimos restrições de chave primária ou estrangeira.

## 3. Criação das tabelas

Criamos o namespace loja_iceberg dentro do catálogo local.

Os comandos DDL definem as tabelas categorias e produtos usando
o formato Iceberg. A propriedade format-version = 2 seleciona
a versão 2 do formato de armazenamento das tabelas.

In [3]:
spark.sql("""
    CREATE NAMESPACE IF NOT EXISTS local.loja_iceberg
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS local.loja_iceberg.categorias (
        id_categoria INT,
        nome_categoria STRING
    )
    USING ICEBERG
    TBLPROPERTIES ('format-version' = '2')
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS local.loja_iceberg.produtos (
        id_produto INT,
        nome_produto STRING,
        preco DECIMAL(10,2),
        estoque INT,
        id_categoria INT
    )
    USING ICEBERG
    TBLPROPERTIES ('format-version' = '2')
""")

spark.sql("SHOW TABLES IN local.loja_iceberg").show()

+------------+----------+-----------+
|   namespace| tableName|isTemporary|
+------------+----------+-----------+
|loja_iceberg|categorias|      false|
|loja_iceberg|  produtos|      false|
+------------+----------+-----------+



## 4. Inserção de dados — INSERT

O conjunto de dados foi criado pelo grupo e contém duas categorias
e dez produtos fictícios, inseridos diretamente por comandos SQL.

Utilizamos os mesmos registros do notebook Delta.

O comando INSERT INTO adiciona os registros às tabelas Iceberg.
As consultas antes e depois permitem verificar o resultado.

In [4]:
print("Antes da inserção:")
spark.sql("SELECT * FROM local.loja_iceberg.categorias").show()
spark.sql("SELECT * FROM local.loja_iceberg.produtos").show()

spark.sql("""
    INSERT INTO local.loja_iceberg.categorias
        (id_categoria, nome_categoria)
    VALUES
        (1, 'Informática'),
        (2, 'Papelaria')
""")

spark.sql("""
    INSERT INTO local.loja_iceberg.produtos
        (id_produto, nome_produto, preco, estoque, id_categoria)
    VALUES
        (1, 'Mouse', 49.90, 20, 1),
        (2, 'Teclado', 120.00, 10, 1),
        (3, 'Caderno', 19.90, 30, 2),
        (4, 'Caneta', 3.50, 100, 2),
        (5, 'Headset', 199.90, 12, 1),
        (6, 'Webcam', 149.90, 8, 1),
        (7, 'Pendrive', 39.90, 25, 1),
        (8, 'Lapiseira', 9.90, 40, 2),
        (9, 'Borracha', 2.50, 60, 2),
        (10, 'Marcador', 6.90, 35, 2)
""")

print("Depois da inserção:")
spark.sql("""
    SELECT * FROM local.loja_iceberg.categorias
    ORDER BY id_categoria
""").show()

spark.sql("""
    SELECT * FROM local.loja_iceberg.produtos
    ORDER BY id_produto
""").show()

Antes da inserção:
+------------+--------------+
|id_categoria|nome_categoria|
+------------+--------------+
+------------+--------------+

+----------+------------+-----+-------+------------+
|id_produto|nome_produto|preco|estoque|id_categoria|
+----------+------------+-----+-------+------------+
+----------+------------+-----+-------+------------+

Depois da inserção:
+------------+--------------+
|id_categoria|nome_categoria|
+------------+--------------+
|           1|   Informática|
|           2|     Papelaria|
+------------+--------------+

+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 49.90|     20|           1|
|         2|     Teclado|120.00|     10|           1|
|         3|     Caderno| 19.90|     30|           2|
|         4|      Caneta|  3.50|    100|           2|
|         5|     Headset|199.90|     12|           1|
|         6|  

## 5. Atualização de dados — UPDATE

O comando UPDATE modifica registros existentes na tabela Iceberg.

Alteramos o preço do Mouse de R$ 49,90 para R$ 59,90 e seu estoque
de 20 para 15 unidades.

A condição WHERE id_produto = 1 limita a atualização ao Mouse.
Os demais produtos mantêm seus valores.

In [5]:
print("Antes da atualização:")
spark.sql("""
    SELECT * FROM local.loja_iceberg.produtos
    ORDER BY id_produto
""").show()

spark.sql("""
    UPDATE local.loja_iceberg.produtos
    SET preco = 59.90,
        estoque = 15
    WHERE id_produto = 1
""")

print("Depois da atualização:")
spark.sql("""
    SELECT * FROM local.loja_iceberg.produtos
    ORDER BY id_produto
""").show()

Antes da atualização:
+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 49.90|     20|           1|
|         2|     Teclado|120.00|     10|           1|
|         3|     Caderno| 19.90|     30|           2|
|         4|      Caneta|  3.50|    100|           2|
|         5|     Headset|199.90|     12|           1|
|         6|      Webcam|149.90|      8|           1|
|         7|    Pendrive| 39.90|     25|           1|
|         8|   Lapiseira|  9.90|     40|           2|
|         9|    Borracha|  2.50|     60|           2|
|        10|    Marcador|  6.90|     35|           2|
+----------+------------+------+-------+------------+

Depois da atualização:
+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 59.90|    

## 6. Exclusão de dados — DELETE

O comando DELETE FROM remove registros da tabela Iceberg.

Excluímos a Caneta, identificada por id_produto = 4.
A condição WHERE limita a exclusão a esse produto.

As consultas antes e depois mostram que os outros nove produtos
permanecem na tabela.

In [6]:
print("Antes da exclusão:")
spark.sql("""
    SELECT * FROM local.loja_iceberg.produtos
    ORDER BY id_produto
""").show()

spark.sql("""
    DELETE FROM local.loja_iceberg.produtos
    WHERE id_produto = 4
""")

print("Depois da exclusão:")
spark.sql("""
    SELECT * FROM local.loja_iceberg.produtos
    ORDER BY id_produto
""").show()

Antes da exclusão:
+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 59.90|     15|           1|
|         2|     Teclado|120.00|     10|           1|
|         3|     Caderno| 19.90|     30|           2|
|         4|      Caneta|  3.50|    100|           2|
|         5|     Headset|199.90|     12|           1|
|         6|      Webcam|149.90|      8|           1|
|         7|    Pendrive| 39.90|     25|           1|
|         8|   Lapiseira|  9.90|     40|           2|
|         9|    Borracha|  2.50|     60|           2|
|        10|    Marcador|  6.90|     35|           2|
+----------+------------+------+-------+------------+

Depois da exclusão:
+----------+------------+------+-------+------------+
|id_produto|nome_produto| preco|estoque|id_categoria|
+----------+------------+------+-------+------------+
|         1|       Mouse| 59.90|     15|  